# Provision an RKE2 Kubernetes Cluster on FABRIC

This notebook builds a 3-node RKE2 cluster for the CSC 478 project:

1. Finds a FABRIC site with enough free cores and RAM.
2. Creates a slice with 3 Ubuntu 22.04 VMs on a private L2 network (`192.168.1.0/24`).
3. Assigns `192.168.1.1`, `.2`, `.3` to `node1`, `node2`, `node3`.
4. Generates the Ansible inventory and playbooks into `../playbook/`.
5. Runs the prerequisite and RKE2 playbooks (node1 = server, node2/node3 = agents).
6. Confirms all nodes are `Ready`.

If a site fails at any step, its slice is deleted and the next site is tried.

**Location:** this notebook must live in `notebooks/`, one level below the project root, so that `../playbook/` points to the project's `playbook/` folder.

## 1. Setup

In [5]:
import shutil
import subprocess
import sys
import time

from ipaddress import IPv4Network
from pathlib import Path

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# utils/ansible.py lives at the root of fabric-examples. Walk up until we find it.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "utils" / "ansible.py").exists():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise ImportError("Could not find repo root containing utils/ansible.py")

REPO_ROOT = _candidate   # the fabric-examples folder

from utils.ansible import generate_rke2_playbooks

fablib = fablib_manager()
fablib.verify_and_configure()

User: EC1053226@wcupa.edu bastion key is valid!
Configuration is valid
User: EC1053226@wcupa.edu bastion key is valid!
Configuration is valid
Please save the config!


## 2. Configuration

In [6]:
SLICE_PREFIX = "echarest-"        # slice name will be SLICE_PREFIX + site name
NETWORK_NAME = "rke2net"
SUBNET = IPv4Network("192.168.1.0/24")

NODES_REQ = 3
CORES_REQ = 8
RAM_REQ   = 10                    # GB
DISK_REQ  = 50                    # GB
IMAGE     = "default_ubuntu_22"

HEADROOM = 2                      # only pick sites with 2x the resources we need

SLICE_TIMEOUT = 20 * 60           # seconds to wait for management IPs before giving up

# generate_rke2_playbooks() always writes here (inside the course folder)...
GENERATED_DIR = REPO_ROOT / "478_examples" / "playbook"
# ...so we copy the fresh files into our own project folder and run them from here.
PLAYBOOK_DIR = Path("../playbook").resolve()
INVENTORY    = PLAYBOOK_DIR / "inventory.yml"
PREREQS      = PLAYBOOK_DIR / "playbook-prereqs.yml"
RKE2         = PLAYBOOK_DIR / "playbook-rke2.yml"

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

GRADER_KEY_PATH = Path("/home/fabric/.ssh/grader_key.pub")

KEEP_FAILED_SLICE = True          # while debugging: stop and keep the slice if Ansible fails
ANSIBLE_WAIT = 5 * 60             # seconds to keep retrying 'ansible ping' before giving up

print("Playbook folder:", PLAYBOOK_DIR)
if not PLAYBOOK_DIR.is_dir():
    raise FileNotFoundError(f"{PLAYBOOK_DIR} not found. Is this notebook in notebooks/?")

Playbook folder: /app/projects/csc478-project/playbook


## 3. Helper functions

In [7]:
def delete_slice_quietly(slice_obj, reason=""):
    """Delete a slice and never raise, so the site loop can keep going."""
    try:
        print(f"Deleting slice {slice_obj.get_name()} {reason}".strip())
        slice_obj.delete()
    except Exception as e:
        print(f"  (delete failed: {e})")


def delete_existing_slice(name):
    """Remove a leftover slice with the same name from an earlier run."""
    for s in fablib.get_slices():
        if s.get_name() == name:
            delete_slice_quietly(s, "(left over from a previous run)")
            return


def build_slice(name, site):
    """Define 3 nodes on a private L2 network. Nothing is submitted yet."""
    slice_obj = fablib.new_slice(name=name)
    net = slice_obj.add_l2network(name=NETWORK_NAME, subnet=SUBNET)
    for i in range(1, NODES_REQ + 1):
        node = slice_obj.add_node(
            name=f"node{i}", site=site,
            cores=CORES_REQ, ram=RAM_REQ, disk=DISK_REQ, image=IMAGE,
        )
        iface = node.add_component(model="NIC_Basic", name="nic").get_interfaces()[0]
        iface.set_mode("config")
        net.add_interface(iface)
    return slice_obj


def wait_for_nodes(slice_obj):
    """Wait until every node has a management IP.
    Returns True on success, False if the slice died or timed out."""
    deadline = time.time() + SLICE_TIMEOUT
    while time.time() < deadline:
        slice_obj.update()
        state = slice_obj.get_state()
        print(f"Slice state: {state}")
        if state in ("Closing", "Dead", "StableError"):
            return False
        nodes = slice_obj.get_nodes()
        if nodes and all(n.get_management_ip() for n in nodes):
            for n in nodes:
                print(f"---- {n.get_name()} ----")
                print("management ip:", n.get_management_ip())
                print(n.get_ssh_command())
            return True
        time.sleep(10)
    print("Timed out waiting for management IPs")
    return False


def configure_dataplane(slice_obj):
    """Bring up the private NIC on each node and assign 192.168.1.N."""
    for i in range(1, NODES_REQ + 1):
        node = slice_obj.get_node(name=f"node{i}")
        iface = node.get_interface(network_name=NETWORK_NAME)
        iface.ip_link_up()
        iface.ip_addr_add(addr=f"192.168.1.{i}", subnet=SUBNET)
        print(f"{node.get_name()} -> 192.168.1.{i}")


def generate_and_copy_playbooks(slice_obj):
    """Generate inventory + playbooks for this slice, then copy them into our project.
    Old files are removed first so a stale inventory can never be used."""
    for f in ("inventory.yml", "playbook-prereqs.yml", "playbook-rke2.yml"):
        (PLAYBOOK_DIR / f).unlink(missing_ok=True)
    generate_rke2_playbooks(slice_obj, fablib, NETWORK_NAME)
    for f in ("inventory.yml", "playbook-prereqs.yml", "playbook-rke2.yml"):
        shutil.copy2(GENERATED_DIR / f, PLAYBOOK_DIR / f)
    print(f"Copied fresh Ansible files into {PLAYBOOK_DIR}")


def wait_for_ansible(log_file):
    """Retry 'ansible all -m ping' until every node answers or we time out."""
    deadline = time.time() + ANSIBLE_WAIT
    attempt = 0
    while time.time() < deadline:
        attempt += 1
        result = subprocess.run(
            ["ansible", "all", "-i", str(INVENTORY), "-m", "ping"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        log_file.write(f"--- ansible ping attempt {attempt} ---\n{result.stdout}\n")
        log_file.flush()
        if result.returncode == 0:
            print(f"Ansible reached all nodes (attempt {attempt})")
            return True
        print(f"Ansible ping attempt {attempt} failed, retrying in 15s")
        time.sleep(15)
    return False


def run_playbook(playbook, log_file):
    """Run one playbook, appending its output to the site's log file."""
    print(f"==== Running {playbook.name} ====")
    result = subprocess.run(
        ["ansible-playbook", "-i", str(INVENTORY), str(playbook)],
        stdout=log_file, stderr=subprocess.STDOUT, text=True,
    )
    return result.returncode == 0


def count_ready_nodes(slice_obj):
    server = slice_obj.get_node("node1")
    stdout, _ = server.execute("kubectl get nodes -o wide", quiet=True)
    print(stdout)
    return sum(1 for line in stdout.splitlines() if " Ready" in line)

## 4. Find a site and build the cluster

This cell tries each site in turn. It can take 15–30 minutes. Ansible output for each attempt goes to `logs/<site>-ansible.log`; check that file if a playbook fails.

In [8]:
resources = fablib.get_resources()
resources.update()

needed_cores = NODES_REQ * CORES_REQ * HEADROOM
needed_ram   = NODES_REQ * RAM_REQ * HEADROOM

grader_keys = []
if GRADER_KEY_PATH.exists():
    grader_keys = [GRADER_KEY_PATH.read_text().strip()]
else:
    print(f"WARNING: {GRADER_KEY_PATH} not found; submitting without the grader key")

slice_obj = None

for site in resources.get_site_names():
    if (resources.get_core_available(site) < needed_cores
            or resources.get_ram_available(site) < needed_ram):
        print(f"{site}: not enough free cores/RAM")
        continue

    name = SLICE_PREFIX + site
    print("=" * 70)
    print(f"Trying {site} (slice {name})")

    delete_existing_slice(name)
    candidate = build_slice(name, site)

    # Submit
    try:
        candidate.submit(progress=False, extra_ssh_keys=grader_keys)
    except Exception as e:
        print(f"{site}: submit failed: {e}")
        delete_slice_quietly(candidate)
        continue

    # Wait for the VMs
    if not wait_for_nodes(candidate):
        delete_slice_quietly(candidate, "(slice did not come up)")
        continue

    # Private network + Ansible
    try:
        configure_dataplane(candidate)
        generate_and_copy_playbooks(candidate)
    except Exception as e:
        print(f"{site}: setup failed: {e}")
        delete_slice_quietly(candidate)
        continue

    log_path = LOG_DIR / f"{site}-ansible.log"
    with open(log_path, "w") as log_file:
        ok = (wait_for_ansible(log_file)
              and run_playbook(PREREQS, log_file)
              and run_playbook(RKE2, log_file))
    if not ok:
        print(f"{site}: Ansible step failed, see {log_path}")
        if KEEP_FAILED_SLICE:
            node1 = candidate.get_node("node1")
            print("Keeping this slice for debugging. Test SSH by hand with:")
            print(f"ssh -v -F /home/fabric/work/fabric_config/ssh_config "
                  f"-i /home/fabric/.ssh/slice_key ubuntu@{node1.get_management_ip()}")
            slice_obj = candidate
            break
        delete_slice_quietly(candidate)
        continue

    # Validate
    ready = count_ready_nodes(candidate)
    if ready == NODES_REQ:
        print(f"{site} works: all {NODES_REQ} nodes Ready")
        slice_obj = candidate
        break

    print(f"{site}: only {ready}/{NODES_REQ} nodes Ready")
    delete_slice_quietly(candidate)
else:
    print("No site succeeded. Check the logs/ folder and try again later.")

Trying CERN (slice echarest-CERN)
Running post boot config threads ...
Post boot config node3, Done! (1 sec)
Post boot config node2, Done! (1 sec)
Post boot config node1, Done! (1 sec)
Saving fablib data...  Done!
Slice state: StableOK
---- node1 ----
management ip: 2001:400:a100:3090:f816:3eff:fe5d:2526
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3090:f816:3eff:fe5d:2526
---- node2 ----
management ip: 2001:400:a100:3090:f816:3eff:fe32:a424
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3090:f816:3eff:fe32:a424
---- node3 ----
management ip: 2001:400:a100:3090:f816:3eff:fe15:2c48
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3090:f816:3eff:fe15:2c48
node1 -> 192.168.1.1
node2 -> 192.168.1.2
node3 -> 192.168.1.3
Wrote Ansible files from /app/478_examples/templates -> /app/478_examples/playbook
Copied fresh Ansible files 

## 5. Reconnect to an existing cluster

If the kernel restarts, run the Setup and Configuration cells, then this one to get the slice back without rebuilding. Set `SITE` to the site that worked.

In [9]:
SITE = ""   # e.g. "STAR"

if SITE:
    slice_obj = fablib.get_slice(name=SLICE_PREFIX + SITE)
    print("Reconnected to", slice_obj.get_name())

## 6. Check the cluster

In [12]:
server = slice_obj.get_node("node1")

for cmd in ["kubectl get nodes -o wide",
            "kubectl get pods -A",
            "curl -s -o /dev/null -w 'nginx-demo HTTP %{http_code}\\n' http://192.168.1.1:30080/"]:
    print(f"$ {cmd}")
    stdout, stderr = server.execute(cmd, quiet=True)
    print(stdout or stderr)

$ kubectl get nodes -o wide
NAME    STATUS   ROLES                AGE     VERSION          INTERNAL-IP   EXTERNAL-IP   OS-IMAGE             KERNEL-VERSION               CONTAINER-RUNTIME
node1   Ready    control-plane,etcd   6m26s   v1.36.4+rke2r1   192.168.1.1   <none>        Ubuntu 22.04.5 LTS   5.15.0-185-generic (amd64)   containerd://2.3.4-k3s1.36
node2   Ready    <none>               4m41s   v1.36.4+rke2r1   192.168.1.2   <none>        Ubuntu 22.04.5 LTS   5.15.0-185-generic (amd64)   containerd://2.3.4-k3s1.36
node3   Ready    <none>               4m38s   v1.36.4+rke2r1   192.168.1.3   <none>        Ubuntu 22.04.5 LTS   5.15.0-185-generic (amd64)   containerd://2.3.4-k3s1.36

$ kubectl get pods -A
NAMESPACE     NAME                                                   READY   STATUS      RESTARTS        AGE
default       nginx-demo-6f8d7bb5d-7fw8s                             1/1     Running     0               3m57s
default       nginx-demo-6f8d7bb5d-lrwdd                          

## 7. SSH commands

In [11]:
for node in slice_obj.get_nodes():
    print(node.get_name())
    print("  ", node.get_ssh_command())

node1
   ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3090:f816:3eff:fe5d:2526
node2
   ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3090:f816:3eff:fe32:a424
node3
   ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3090:f816:3eff:fe15:2c48


## 8. Delete the slice

Only run this when you are done with the cluster.

In [ ]:
DELETE = False   # set to True to delete

if DELETE and slice_obj:
    slice_obj.delete()
    print("Slice deleted")